In [1]:
DATA = "bookings.jsonl"

# Task 2: Contracts, nested data and a star schema

**Pipeline overview**

1. **Part A – Contract:** validate every JSON line against Pydantic models; failures (and re-sent duplicates) go to `quarantine.jsonl` with a reason.
2. **Part B – Flatten:** turn the valid nested documents into two flat tables, `bookings` (one row per booking) and `booking_tickets` (one row per ticket entry); detect schema drift.
3. **Part C – Model & store:** build a star schema in DuckDB (`dim_event`, `dim_student`, `fact_ticket_sales`), answer a business question from it, and save it as Parquet under `star/`.

The first cell above sets `DATA`; every cell below reads only from `DATA`, so the notebook can be re-run on a different file. No count is typed in by hand — every number is computed.

In [2]:
# Imports used throughout the notebook
import json
import math
import shutil
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Literal, Optional

import duckdb
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, model_validator

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Part A: The contract

### A.1 Pydantic models

One model per nested object (`Ticket`, `Student`, `Event`) and a `Booking` model that uses all three. Each rule from the contract maps to a type or constraint:

| Rule | How it is enforced |
|---|---|
| `student.id` = two digits, dash, four digits | `Field(pattern=r"^\d{2}-\d{4}$")` |
| `student.faculty`, `event.club`, `ticket.type`, `payment`, `status` | `Literal[...]` – only the listed values are accepted |
| `event` present with all five fields | `event: Event` is required (a `null` event fails); every `Event` field is required and non-empty |
| at least one ticket | `Field(min_length=1)` on the list |
| `qty` ≥ 1, `price` ≥ 0 | `Field(ge=1)` / `Field(ge=0)` |
| `total` = Σ qty × price **(cross-field)** | `model_validator(mode="after")` |
| `booked_at` before the event's date **(cross-field)** | same `model_validator` |

The document also has an optional `promo_code` field that is not in the README example (schema drift, see Part B.3). It is declared as `Optional[str] = None`, so documents with *or without* it are both accepted and it is kept for the flat tables.

In [3]:
# --- Allowed values (taken from the contract) ------------------------------
Faculty     = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
Club        = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TicketType  = Literal["standard", "vip", "student"]
Payment     = Literal["card", "cash", "wallet"]
Status      = Literal["confirmed", "cancelled"]

MONEY_TOL = 0.005   # half a piastre: any difference smaller than this is only float rounding noise


class Ticket(BaseModel):
    """One entry of the `tickets` list."""
    type: TicketType                                  # standard / vip / student only
    qty: int = Field(ge=1)                            # at least one ticket of this type
    price: float = Field(ge=0, allow_inf_nan=False)   # EGP per ticket, never negative


class Student(BaseModel):
    """Who made the booking."""
    id: str = Field(pattern=r"^\d{2}-\d{4}$")         # e.g. 58-1234
    name: str
    faculty: Faculty


class Event(BaseModel):
    """The event being booked. All five fields are required and must not be empty."""
    id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    club: Club
    date: date
    venue: str = Field(min_length=1)


class Booking(BaseModel):
    """A whole booking document; uses the three models above."""
    booking_id: str = Field(min_length=1)
    booked_at: datetime
    student: Student
    event: Event                                       # required: a missing / null event fails
    tickets: list[Ticket] = Field(min_length=1)        # at least one ticket entry
    total: float = Field(allow_inf_nan=False)
    payment: Payment
    status: Status
    promo_code: Optional[str] = None                   # drifting field: optional, defaults to None

    @model_validator(mode="after")
    def cross_field_rules(self):
        """The two rules that need more than one field. Runs only after every field passed."""
        problems = []

        # Rule 1: total must equal the sum of qty * price over all ticket entries
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=MONEY_TOL):
            problems.append("total does not equal sum of qty x price")

        # Rule 2: the booking must be made before the event's date.
        # booked_at is in UTC; a timestamp without a timezone is treated as UTC.
        booked = self.booked_at if self.booked_at.tzinfo else self.booked_at.replace(tzinfo=timezone.utc)
        if not booked.astimezone(timezone.utc).date() < self.event.date:
            problems.append("booked_at is not before the event date")

        if problems:
            raise ValueError("; ".join(problems))
        return self

### A.2 + A.3 Validate every line, then remove re-sent bookings

* Each line is parsed as JSON and validated with `Booking.model_validate`. A line that is not valid JSON, or that breaks any rule, goes to the quarantine with its `booking_id` and a **reason**.
* Pydantic error messages are turned into short, stable reasons like `student.id: String should match pattern ...` (list indices such as `tickets.0.qty` are dropped to `tickets.qty`) so that identical problems are counted together.
* **Duplicates are checked after validation:** among the bookings that passed, the first copy of each `booking_id` is kept and every later copy is quarantined with the reason `duplicate booking_id`. Checking after validation means an invalid first copy can never "win" and push out a later valid copy.
* `quarantine.jsonl` holds one object per quarantined line, with only `booking_id` and `reason`: contract failures first (in file order), then duplicates.

In [4]:
def describe_error(err: dict) -> str:
    """Turn one pydantic error dict into a short, value-free reason string."""
    # location without list indices: ('tickets', 0, 'qty') -> 'tickets.qty'; model-level -> 'booking'
    loc = ".".join(str(p) for p in err["loc"] if not isinstance(p, int)) or "booking"
    etype = err["type"]
    if etype == "missing":
        msg = "is missing"
    elif etype in ("model_type", "model_attributes_type") and err.get("input") is None:
        msg = "is missing (null)"
    elif etype == "value_error":                      # raised by our model_validator
        msg = err["msg"].removeprefix("Value error, ")
    else:
        msg = err["msg"]
    return f"{loc}: {msg}"


def reason_from(exc: ValidationError) -> str:
    """All distinct problems of one document, joined in a stable order."""
    return "; ".join(sorted({describe_error(e) for e in exc.errors()}))


raw_docs = []      # (line_no, parsed dict) for every line that is valid JSON – used for drift detection
passed = []        # Booking objects that met the contract, in file order
quarantine = []    # dicts: booking_id, reason
lines_read = 0

with open(DATA, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():            # ignore blank lines (e.g. a trailing newline)
            continue
        lines_read += 1

        # 1) parse JSON
        try:
            doc = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"booking_id": None, "reason": "invalid JSON"})
            continue
        if isinstance(doc, dict):
            raw_docs.append((line_no, doc))
        bid = doc.get("booking_id") if isinstance(doc, dict) else None

        # 2) validate against the contract
        try:
            passed.append(Booking.model_validate(doc))
        except ValidationError as exc:
            quarantine.append({"booking_id": bid, "reason": reason_from(exc)})

# 3) duplicates – AFTER validation: keep the first valid copy of each booking_id
seen = set()
valid = []         # the final, clean list of Booking objects
for b in passed:
    if b.booking_id in seen:
        quarantine.append({"booking_id": b.booking_id, "reason": "duplicate booking_id"})
    else:
        seen.add(b.booking_id)
        valid.append(b)

# save the quarantine: one JSON object per line, with booking_id and reason only
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print(f"Wrote {len(quarantine)} quarantined lines to quarantine.jsonl")

Wrote 24 quarantined lines to quarantine.jsonl


### A.4 Report

Lines read, lines that passed, and the number of quarantined lines **per reason** – all computed from the lists above.

In [5]:
reason_counts = (pd.Series([q["reason"] for q in quarantine], name="reason", dtype="object")
                   .value_counts()
                   .rename_axis("reason")
                   .reset_index(name="lines"))

print(f"Lines read            : {lines_read}")
print(f"Passed (valid, unique): {len(valid)}")
print(f"Quarantined           : {len(quarantine)}")
assert lines_read == len(valid) + len(quarantine), "every line must end up either valid or quarantined"
reason_counts

Lines read            : 283
Passed (valid, unique): 259
Quarantined           : 24


,reason,lines
0,student.id: String should match pattern '^\d{2...,4
1,booking: total does not equal sum of qty x price,4
2,"tickets.type: Input should be 'standard', 'vip...",3
3,tickets.qty: Input should be greater than or e...,3
4,booking: booked_at is not before the event date,3
5,duplicate booking_id,3
6,event: is missing (null),2
7,tickets: List should have at least 1 item afte...,2


In [6]:
# The quarantined lines themselves (booking_id + reason), for inspection
pd.DataFrame(quarantine, columns=["booking_id", "reason"])

,booking_id,reason
0,B0011,"tickets.type: Input should be 'standard', 'vip..."
1,B0016,tickets.qty: Input should be greater than or e...
2,B0024,student.id: String should match pattern '^\d{2...
3,B0049,event: is missing (null)
4,B0057,student.id: String should match pattern '^\d{2...
5,B0060,student.id: String should match pattern '^\d{2...
6,B0090,"tickets.type: Input should be 'standard', 'vip..."
7,B0101,booking: total does not equal sum of qty x price
8,B0106,student.id: String should match pattern '^\d{2...
9,B0116,booking: booked_at is not before the event date


## Part B: Flatten

### B.1 Two flat tables from the valid bookings

* **`bookings`** – one row per booking. `pd.json_normalize(..., sep="_")` turns the nested `student` and `event` objects into columns such as `student_id`, `event_club`; the `tickets` list is dropped.
* **`booking_tickets`** – one row per ticket entry. `json_normalize(record_path="tickets", meta=["booking_id"])` "explodes" the list and carries the parent `booking_id` down; `amount = qty × price`.

The data is taken from the **validated** models (`model_dump`), so types are already clean (`booked_at` is a real datetime, `event_date` a real date, a missing `promo_code` is `None`).

In [7]:
records = [b.model_dump() for b in valid]

# bookings: one row per booking, nested objects become prefixed columns, no tickets column
bookings = pd.json_normalize(records, sep="_").drop(columns=["tickets"])
# store booked_at as a timezone-free UTC timestamp (it is UTC by contract) and event_date as a date
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True).dt.tz_convert(None)
bookings["event_date"] = pd.to_datetime(bookings["event_date"])
bookings = bookings[[
    "booking_id", "booked_at",
    "student_id", "student_name", "student_faculty",
    "event_id", "event_title", "event_club", "event_date", "event_venue",
    "total", "payment", "status", "promo_code",
]]

# booking_tickets: one row per entry in the tickets list, with the parent booking_id
booking_tickets = pd.json_normalize(records, record_path="tickets", meta=["booking_id"])
booking_tickets = booking_tickets[["booking_id", "type", "qty", "price"]]
booking_tickets["amount"] = booking_tickets["qty"] * booking_tickets["price"]

print("bookings       :", bookings.shape)
print("booking_tickets:", booking_tickets.shape)
display(bookings.head())
display(booking_tickets.head())

bookings       : (259, 14)
booking_tickets: (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,NaN
1,B0002,2026-09-27 12:29:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,NaN
2,B0003,2026-10-13 20:19:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,NaN
3,B0004,2026-11-01 06:42:00,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,105.0,card,confirmed,CLUB10
4,B0005,2026-10-28 19:06:00,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,20.0,cash,confirmed,NaN


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


### B.2 Checks with assertions

1. `booking_id` is unique in `bookings` (the grain is one row per booking).
2. For every booking, the `amount`s of its ticket rows add up to its `total` (no ticket rows lost or duplicated while flattening).

In [8]:
# 1) grain of bookings: booking_id is unique
assert bookings["booking_id"].is_unique, "booking_id is not unique in bookings"

# 2) per booking, sum(amount) of its tickets == total
amount_per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"].to_frame().join(amount_per_booking)
assert check["amount"].notna().all(), "a booking has no ticket rows"
assert ((check["total"] - check["amount"]).abs() <= MONEY_TOL).all(), "ticket amounts do not add up to total"
assert set(booking_tickets["booking_id"]) == set(bookings["booking_id"]), "orphan ticket rows"

print(f"OK – {len(bookings)} unique bookings, {len(booking_tickets)} ticket rows, all amounts add up to total")

OK – 259 unique bookings, 314 ticket rows, all amounts add up to total


### B.3 Schema drift

The README example has a fixed set of keys (top level and inside `student`, `event` and each ticket). The cell below collects every key **path** that actually appears in the raw documents, compares it with the example, and for each extra key reports how many documents carry it and **when it first appears** (earliest `booked_at`, plus the first line in the file).

In [9]:
# Key paths of the README example document
EXPECTED_KEYS = {
    "booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status",
    "student.id", "student.name", "student.faculty",
    "event.id", "event.title", "event.club", "event.date", "event.venue",
    "tickets[].type", "tickets[].qty", "tickets[].price",
}

def key_paths(doc: dict) -> set:
    """All key paths in one document, e.g. 'student.id', 'tickets[].qty'."""
    paths = set()
    for k, v in doc.items():
        paths.add(k)
        if isinstance(v, dict):
            paths |= {f"{k}.{kk}" for kk in v}
        elif isinstance(v, list):
            paths |= {f"{k}[].{kk}" for item in v if isinstance(item, dict) for kk in item}
    return paths

drift_rows = []
for line_no, doc in raw_docs:
    ts = pd.to_datetime(doc.get("booked_at"), utc=True, errors="coerce")
    for p in key_paths(doc) - EXPECTED_KEYS:
        drift_rows.append({"field": p, "line": line_no, "booking_id": doc.get("booking_id"), "booked_at": ts})
drift = pd.DataFrame(drift_rows, columns=["field", "line", "booking_id", "booked_at"])

if drift.empty:
    print("No schema drift: every document has only the README fields.")
else:
    raw_times = pd.Series([pd.to_datetime(d.get("booked_at"), utc=True, errors="coerce") for _, d in raw_docs])
    summary = []
    for field, g in drift.groupby("field"):
        first = g.sort_values("booked_at").iloc[0]
        has_field = set(g["line"])
        # documents booked at/after the first appearance that still do NOT have the field
        later_without = sum(1 for (ln, _), t in zip(raw_docs, raw_times)
                            if ln not in has_field and pd.notna(t) and t >= first["booked_at"])
        summary.append({
            "field": field,
            "documents_with_field": len(g),
            "documents_without_field": len(raw_docs) - len(g),
            "first_booked_at (UTC)": first["booked_at"],
            "first_booking_id": first["booking_id"],
            "first_line_in_file": int(g["line"].min()),
            "later_docs_without_it": later_without,
        })
    drift_summary = pd.DataFrame(summary)
    display(drift_summary)

,field,documents_with_field,documents_without_field,first_booked_at (UTC),first_booking_id,first_line_in_file,later_docs_without_it
0,promo_code,193,90,2026-10-20 01:05:00+00:00,B0125,4,1


## Part C: Model and store (DuckDB star schema)

### C.1 Build the star

* **`dim_event`** – one row per `event_id`, surrogate key `event_key` (1, 2, 3 … via `ROW_NUMBER()`).
* **`dim_student`** – one row per `student_id`, surrogate key `student_key`.
* **`fact_ticket_sales`** – grain: **one row per ticket entry** (same grain as `booking_tickets`). Holds the foreign keys `event_key`, `student_key`, the measures `qty`, `price`, `amount`, the booking's `status`, and `booked_date_key` = `booked_at` as an integer `YYYYMMDD`.

Each dimension is grouped by its natural id, so it always has exactly one row per id. Its descriptive columns come from the id's most recent booking (`arg_max(..., booked_at)`), and `ROW_NUMBER()` adds the surrogate key.

In [10]:
con = duckdb.connect()                     # in-memory DuckDB database


def run(sql: str) -> list:
    """Execute one SQL statement and fetch ALL of its result rows.
    DuckDB 2.0 blocks the connection until a result is fully read, so every
    statement goes through here; run(q)[0][0] gives a single value."""
    return con.execute(sql).fetchall()


con.register("bookings_df", bookings)      # expose the pandas tables to SQL
con.register("booking_tickets_df", booking_tickets)
run("CREATE OR REPLACE TABLE bookings        AS SELECT * FROM bookings_df")
run("CREATE OR REPLACE TABLE booking_tickets AS SELECT * FROM booking_tickets_df")

# ---- dim_event: one row per event, surrogate key event_key
run("""
CREATE OR REPLACE TABLE dim_event AS
SELECT ROW_NUMBER() OVER (ORDER BY event_id) AS event_key, *
FROM (
    SELECT event_id,
           arg_max(event_title, booked_at)              AS title,
           arg_max(event_club,  booked_at)              AS club,
           CAST(arg_max(event_date, booked_at) AS DATE) AS event_date,
           arg_max(event_venue, booked_at)              AS venue
    FROM bookings GROUP BY event_id
)
""")

run("""
CREATE OR REPLACE TABLE dim_student AS
SELECT ROW_NUMBER() OVER (ORDER BY student_id) AS student_key, *
FROM (
    SELECT student_id,
           arg_max(student_name,    booked_at) AS name,
           arg_max(student_faculty, booked_at) AS faculty
    FROM bookings GROUP BY student_id
)
""")

# ---- fact_ticket_sales: one row per ticket entry
run("""
CREATE OR REPLACE TABLE fact_ticket_sales AS
SELECT bt.booking_id,
       de.event_key,
       ds.student_key,
       bt.type                                         AS ticket_type,
       bt.qty,
       bt.price,
       bt.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS booked_date_key
FROM booking_tickets bt
JOIN bookings    b  ON b.booking_id  = bt.booking_id
JOIN dim_event   de ON de.event_id   = b.event_id
JOIN dim_student ds ON ds.student_id = b.student_id
""")

for t in ["dim_event", "dim_student", "fact_ticket_sales"]:
    print(f"{t:18s} rows = {run(f'SELECT COUNT(*) FROM {t}')[0][0]}")
display(con.sql("SELECT * FROM dim_event").df())
display(con.sql("SELECT * FROM dim_student LIMIT 5").df())
display(con.sql("SELECT * FROM fact_ticket_sales LIMIT 5").df())

dim_event          rows = 8
dim_student        rows = 103
fact_ticket_sales  rows = 314


,event_key,event_id,title,club,event_date,venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


,student_key,student_id,name,faculty
0,1,58-1324,Nour Fouad,MET
1,2,58-1449,Farida Mansour,EMS
2,3,58-1801,Karim Kamel,IET
3,4,58-2165,Farida Hassan,Management
4,5,58-2307,Farida Hassan,Pharmacy


,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,booked_date_key
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101


### C.2 Checks with assertions

1. The fact has exactly as many rows as `booking_tickets` (the joins neither dropped nor multiplied rows).
2. `SUM(amount)` in the fact equals `SUM(total)` in `bookings` (no money lost or double-counted).

Extra sanity checks: each dimension has exactly one row per natural id, and no fact row has a `NULL` key.

In [11]:
fact_rows    = run("SELECT COUNT(*) FROM fact_ticket_sales")[0][0]
fact_amount  = run("SELECT SUM(amount) FROM fact_ticket_sales")[0][0]
total_amount = run("SELECT SUM(total) FROM bookings")[0][0]

# 1) same grain as booking_tickets
assert fact_rows == len(booking_tickets), f"fact has {fact_rows} rows, booking_tickets has {len(booking_tickets)}"
# 2) money is preserved
assert math.isclose(fact_amount, total_amount, abs_tol=MONEY_TOL), f"{fact_amount} != {total_amount}"

# extra: one row per natural id in each dimension, and every fact row found its keys
assert run("SELECT COUNT(*) = COUNT(DISTINCT event_id) FROM dim_event")[0][0]
assert run("SELECT COUNT(*) = COUNT(DISTINCT student_id) FROM dim_student")[0][0]
assert run("SELECT COUNT(*) FROM fact_ticket_sales WHERE event_key IS NULL OR student_key IS NULL")[0][0] == 0

print(f"OK – fact rows = {fact_rows} = booking_tickets rows; SUM(amount) = {fact_amount:,.2f} = SUM(total) = {total_amount:,.2f}")

OK – fact rows = 314 = booking_tickets rows; SUM(amount) = 31,285.00 = SUM(total) = 31,285.00


#### Why `total` is not a fact column

`total` belongs to the **booking** grain, but the fact's grain is the **ticket entry**. A booking with two ticket types has two fact rows; putting `total` on both would repeat it, so `SUM(total)` over the fact would count such bookings twice. 

In [12]:
wrong = run("""
    SELECT SUM(b.total) FROM fact_ticket_sales f JOIN bookings b USING (booking_id)
""")[0][0]
multi = run("""
    SELECT COUNT(*) FROM (SELECT booking_id FROM fact_ticket_sales GROUP BY booking_id HAVING COUNT(*) > 1)
""")[0][0]
print(f"Correct revenue  SUM(amount) over fact          : {fact_amount:,.2f}")
print(f"Wrong revenue    SUM(total) repeated on fact rows: {wrong:,.2f}")
print(f"Over-statement                                   : {wrong - fact_amount:,.2f}  "
      f"(from {multi} bookings with more than one ticket entry)")

Correct revenue  SUM(amount) over fact          : 31,285.00
Wrong revenue    SUM(total) repeated on fact rows: 42,000.00
Over-statement                                   : 10,715.00  (from 55 bookings with more than one ticket entry)


### C.3 Answer from the star

For **confirmed** bookings only: number of tickets sold (`SUM(qty)`) and revenue (`SUM(amount)`) per **club** and **ticket type**. The club comes from `dim_event`, joined through the surrogate key.

In [13]:
answer = con.sql("""
SELECT e.club,
       f.ticket_type,
       SUM(f.qty)    AS tickets_sold,
       SUM(f.amount) AS revenue_egp
FROM fact_ticket_sales f
JOIN dim_event e ON e.event_key = f.event_key
WHERE f.status = 'confirmed'
GROUP BY e.club, f.ticket_type
ORDER BY e.club, f.ticket_type
""").df()
answer

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


### C.4 Save as Parquet

* `star/dim_event.parquet` and `star/dim_student.parquet` – one file per dimension.
* `star/fact_ticket_sales/booked_month=YYYY-MM/…parquet` – the fact, **partitioned by the month the booking was made** (Hive-style folders).

The `star/` folder is deleted first so a re-run (or a run on another file) never leaves stale partitions behind. The files are then read back to prove they are complete.

In [14]:
STAR = Path("star")
if STAR.exists():
    shutil.rmtree(STAR)            # start clean: no stale partitions from earlier runs
STAR.mkdir()

run("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT PARQUET)")
run("COPY dim_student TO 'star/dim_student.parquet' (FORMAT PARQUET)")

# booked_month derived from the YYYYMMDD date key, e.g. 20261027 -> '2026-10'
run("""
COPY (
    SELECT *,
           substr(CAST(booked_date_key AS VARCHAR), 1, 4) || '-' ||
           substr(CAST(booked_date_key AS VARCHAR), 5, 2) AS booked_month
    FROM fact_ticket_sales
) TO 'star/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))
""")

# show what was written
for p in sorted(STAR.rglob("*.parquet")):
    print(p.as_posix())

# read back and verify nothing was lost
back = run("""
    SELECT COUNT(*), SUM(amount)
    FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)
""")[0]
assert back[0] == fact_rows and math.isclose(back[1], fact_amount, abs_tol=MONEY_TOL)
assert run("SELECT COUNT(*) FROM 'star/dim_event.parquet'")[0][0] == \
       run("SELECT COUNT(*) FROM dim_event")[0][0]
assert run("SELECT COUNT(*) FROM 'star/dim_student.parquet'")[0][0] == \
       run("SELECT COUNT(*) FROM dim_student")[0][0]
print(f"Read back {back[0]} fact rows, SUM(amount) = {back[1]:,.2f} – matches the DuckDB tables")

con.sql("""
    SELECT booked_month, COUNT(*) AS fact_rows
    FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)
    GROUP BY booked_month ORDER BY booked_month
""").df()

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet
Read back 314 fact rows, SUM(amount) = 31,285.00 – matches the DuckDB tables


,booked_month,fact_rows
0,2026-09,12
1,2026-10,179
2,2026-11,123
